In [ ]:
!pwd


In [ ]:
!nvidia-smi


# Create Environment


In [ ]:
!test -d /content/edge-lipsync-model || git clone https://github.com/monkira99/edge-lipsync-model.git /content/edge-lipsync-model


In [ ]:
%cd /content/edge-lipsync-model


In [ ]:
!git pull


In [ ]:
!uv sync


In [ ]:
import os

try:
    from google.colab import userdata
    for key in ("HF_TOKEN", "WANDB_API_KEY"):
        value = userdata.get(key)
        if value:
            os.environ[key] = value
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or environment."
assert os.environ.get("WANDB_API_KEY"), "Set WANDB_API_KEY in Colab Secrets or environment."


In [ ]:
!hf auth login --token "$HF_TOKEN" --quiet
!uv run wandb login "$WANDB_API_KEY" --verify


In [ ]:
!uv run python tools/hf_model_assets.py pull \
  --repo-id tiennguyenbnbk/edge-lipsync-model-assets \
  --local-dir models


In [ ]:
!ls -lah /content/edge-lipsync-model/models/


# Training Configuration

Train directly from a pinned, self-contained silent/talking pose-paired snapshot. Set `HF_DATASET_REVISION` to the full revision SHA printed by `notebooks/build_datasets.ipynb`.


In [ ]:
from pathlib import Path
import json

ROOT = Path("/content/edge-lipsync-model")
HF_DATASET_REPO = "tiennguyenbnbk/nora-silent-talking-pose-pairs"
HF_DATASET_REVISION = ""  # Paste the full revision SHA from notebooks/build_datasets.ipynb.
HF_MODEL_REPO = "tiennguyenbnbk/lipsync-nora-silent-talking-v1"

RUN_DIR = Path("/content/data/runs/nora_silent_talking_v1")
HF_CACHE_DIR = Path("/content/data/hf_cache")
WANDB_DIR = Path("/content/data/wandb")
HF_DATASET_LOCAL_DIR = Path("/content/data/hf_snapshots") / HF_DATASET_REPO.split("/")[-1] / (HF_DATASET_REVISION or "set-HF_DATASET_REVISION")
TRAIN_CONFIG_PATH = ROOT / "configs/train.colab.yaml"

assert HF_DATASET_REVISION, "Set HF_DATASET_REVISION to the full commit SHA from the build notebook push-snapshot output."

for directory in (RUN_DIR, HF_CACHE_DIR, WANDB_DIR, HF_DATASET_LOCAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"dataset_repo={HF_DATASET_REPO}")
print(f"dataset_revision={HF_DATASET_REVISION}")
print(f"dataset_local_dir={HF_DATASET_LOCAL_DIR}")
print(f"run_dir={RUN_DIR}")
print(f"model_repo={HF_MODEL_REPO}")


In [ ]:
train_config = f"""run_dir: {RUN_DIR}

hf_dataset_repo: "{HF_DATASET_REPO}"
hf_dataset_revision: "{HF_DATASET_REVISION}"
hf_dataset_local_dir: "{HF_DATASET_LOCAL_DIR}"
hf_cache_dir: "{HF_CACHE_DIR}"

init_bin: {ROOT / 'models/emma/dh_model.bin'}
init_ckpt: ""
hf_init_model_repo: ""
hf_init_model_filename: best.pt

hf_model_repo: "{HF_MODEL_REPO}"
hf_model_private: true

device: cuda
precision: mixed
batch_size: 64
num_workers: 4
learning_rate: 0.00001
weight_decay: 0.0001

max_steps: 10000
warmup_steps: 100
stabilization_steps: 10
stabilization_lr_scale: 0.1
validation_interval: 20
checkpoint_interval: 20
log_interval: 10
media_eval_on_best: true
media_eval_clip_count: 2
media_eval_clip_ids: []
media_eval_max_frames_per_clip: 250
media_eval_fps: 25.0
media_eval_log_to_wandb: false

wandb_mode: online
wandb_project: edge-lipsync-model
wandb_entity: ""
wandb_run_name: nora-silent-talking-v1
wandb_group: nora-silent-talking
wandb_tags:
  - silent-talking
  - pose-paired
  - nora
  - duix-unet
wandb_notes: "Fine-tuning from a pinned silent/talking pose-paired snapshot"
wandb_dir: {WANDB_DIR}
"""

TRAIN_CONFIG_PATH.write_text(train_config, encoding="utf-8")
print(TRAIN_CONFIG_PATH)


In [ ]:
!sed -n "1,240p" {TRAIN_CONFIG_PATH}


In [ ]:
!uv run python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')"


# Pull And Verify Dataset Snapshot


In [ ]:
!uv run python tools/hf_dataset.py pull-snapshot \
  --repo-id {HF_DATASET_REPO} \
  --revision {HF_DATASET_REVISION} \
  --local-dir {HF_DATASET_LOCAL_DIR} \
  --cache-dir {HF_CACHE_DIR}


In [ ]:
from datasets import Image, load_from_disk
from torch.utils.data import DataLoader

from edge_lipsync.dataset import DuixHFDataset
from edge_lipsync.training import collate_training_batch, validate_batch_shapes

dataset = load_from_disk(HF_DATASET_LOCAL_DIR / "dataset")
train_dataset = DuixHFDataset(dataset, "train")
val_dataset = DuixHFDataset(dataset, "val")
batch = collate_training_batch([train_dataset[0], val_dataset[0]])
validate_batch_shapes(batch)
raw_roi = dataset["train"].cast_column("source_roi", Image(decode=False))[0]["source_roi"]

print({"train": len(train_dataset), "val": len(val_dataset)})
print(batch["face"].shape, batch["audio"].shape, batch["target"].shape)
print(batch["meta"][0])
print(f"source_roi_path={raw_roi['path']}")
print(f"source_roi_png_magic={raw_roi['bytes'][:8]}")


# Preview Validation Pair


In [ ]:
from IPython.display import display

row = dataset["val"][0]
print({
    "pair_id": row["pair_id"],
    "source_frame_idx": row["source_frame_idx"],
    "target_frame_idx": row["target_frame_idx"],
    "matching_score": row["matching_score"],
    "sync_confidence": row["sync_confidence"],
    "sample_weight": row["sample_weight"],
})
display(row["source_roi"])
display(row["target_roi"])


# Train With Console And W&B Logging


In [ ]:
!PYTHONUNBUFFERED=1 uv run python -u tools/train.py --config {TRAIN_CONFIG_PATH}


In [ ]:
metrics_csv = RUN_DIR / "metrics.csv"
!tail -n 20 {metrics_csv} || true

from IPython.display import Video, display

metadata_path = RUN_DIR / "run_metadata.json"
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    print(metadata.get("provenance", {}).get("dataset", {}))
    print(metadata.get("provenance", {}).get("wandb", {}))
    print(metadata.get("provenance", {}).get("media_eval", {}))

media_eval_index = RUN_DIR / "media_eval" / "index.json"
if media_eval_index.exists():
    media_rows = json.loads(media_eval_index.read_text())
    latest = media_rows[-1]
    print(latest)
    display(Video(latest["video_path"], embed=True))
else:
    print("No media_eval/index.json yet. It is created when validation produces a new best.pt.")


# Optional Push Model Artifacts


In [ ]:
!uv run python tools/hf_model.py push \
  --run-dir {RUN_DIR} \
  --repo-id {HF_MODEL_REPO}
